In [ ]:
# Run once if the package is not already installed in editable mode.
!pip install -e ".[dev]"


In [ ]:
from copy import deepcopy

from semantic_kv.eviction import LRUEviction, SemanticEviction
from semantic_kv.models import MODEL_PRESETS
from semantic_kv.placement import NaiveHBMPolicy, SemanticKVPolicy
from semantic_kv.simulator import SimulationEngine
from semantic_kv.tiers import default_tier_profiles
from semantic_kv.workloads import shared_prefix_workload

profile = MODEL_PRESETS["llama8b"]
workload = shared_prefix_workload(
    profile,
    sessions=20,
    context=4096,
    decode_steps=32,
)

semantic_engine = SimulationEngine(
    model_profile=profile,
    workload=deepcopy(workload),
    placement_policy=SemanticKVPolicy(),
    eviction_policy=SemanticEviction(),
    tier_config=default_tier_profiles(scale=0.02),
)
semantic_metrics = semantic_engine.run()
semantic_metrics.as_row("Semantic KV")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(semantic_metrics.occupancy_history)
plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(10, 5))
for column in ["GPU_HBM", "KV_APPLIANCE", "CXL_POOL", "NVME_OBJECT"]:
    ax.plot(history["step"], history[column] / (1024 ** 3), label=column)
ax.set_title("Tier Occupancy Over Decode Steps")
ax.set_xlabel("Step")
ax.set_ylabel("Used capacity (GB)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


In [ ]:
naive_engine = SimulationEngine(
    model_profile=profile,
    workload=deepcopy(workload),
    placement_policy=NaiveHBMPolicy(),
    eviction_policy=LRUEviction(),
    tier_config=default_tier_profiles(scale=0.02),
)
naive_metrics = naive_engine.run()

comparison = pd.DataFrame(
    [
        {
            "policy": "Naive HBM",
            "peak_hbm_gb": naive_metrics.hbm_used_peak / (1024 ** 3),
            "throughput_score": naive_metrics.estimated_throughput_score,
        },
        {
            "policy": "Semantic KV",
            "peak_hbm_gb": semantic_metrics.hbm_used_peak / (1024 ** 3),
            "throughput_score": semantic_metrics.estimated_throughput_score,
        },
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparison.plot.bar(x="policy", y="peak_hbm_gb", ax=axes[0], color=["#38bdf8", "#22c55e"])
comparison.plot.bar(x="policy", y="throughput_score", ax=axes[1], color=["#38bdf8", "#22c55e"])
axes[0].set_title("Peak HBM Usage")
axes[0].set_ylabel("GB")
axes[1].set_title("Throughput Proxy")
axes[1].set_ylabel("Score")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Extending with a Custom Policy Class

To add a new placement strategy, subclass `PlacementPolicy` and implement
`choose_tier(block, tiers)`. Reuse the existing `SimulationEngine` and
`make_workload(...)` helpers so your experiment stays comparable to the
built-in policies. If the policy needs custom eviction semantics, pair it
with a matching `EvictionPolicy` implementation and run both against the
same workload snapshots.